# 19 — Testing and debugging patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/python/19-testing-and-debugging/lab.ipynb) [![Open in LinguAgent](https://img.shields.io/badge/Open%20in-LinguAgent-1f6f5c)](https://linguagent.com/courses/python/testing-and-debugging?utm_source=github&utm_medium=labs)

*~45 minutes · 23 tests*

Every other module in this repo asks you to make tests pass. This one is
about **writing** them — the techniques that make a test suite actually
useful: mocking out the parts you don't control, injecting a fake clock
instead of sleeping, capturing logs instead of scraping stdout, and turning
"debug this by staring at it" into a search you can automate.

`test_exercises.py` in this module is itself the primary teaching material
for two of the seven sections — read it, not just the README.

Work in `exercises.py`.

**How this notebook works:** run the first cell; then, section by section, read, write the code in the cell under it, and run the check below it. Stuck? `lab.solution("name")` shows the reference — try first, it's worth it.

In [ ]:
#@title ▶️ Run me first — fetches this lab and its checker
import os, sys, subprocess
HOME = "/content/linguagent-labs"
if "google.colab" in sys.modules:
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/mrshahabian/linguagent-labs.git", HOME], check=True)
else:  # Jupyter, VS Code or a Codespace: this notebook already sits inside the repo
    HOME = os.path.abspath(os.path.join(os.getcwd(), *[".."] * 3)) if os.path.isdir("../../../labkit") else os.getcwd()
sys.path.insert(0, HOME)
import labkit
lab = labkit.open("python/testing-and-debugging")

In [ ]:
# What the exercises below use — run this cell.
import logging

import time

logger = logging.getLogger(__name__)

class MatrixShapeError(ValueError):
    """Raised when two matrices' shapes are incompatible for multiplication."""

    pass

## What you'll build

| Name | You'll implement | Shows up in real code as |
|---|---|---|
| `is_prime` | `is_prime` | The subject of a table-driven test using `subTest` |
| `fetch_first_success` | `fetch_first_success` | A function shaped for mocking — the client is a parameter, not an import |
| `TTLCache` | `set`, `get` | A time-based cache tested with a fake clock instead of `time.sleep` |
| `divide_safely` | `divide_safely` | Logging a warning instead of printing, so tests can assert on it |
| `find_first_failure` | `find_first_failure` | `git bisect`'s algorithm, generalised to any monotonic predicate |
| `MatrixShapeError`, `matmul_checked` | `matmul_checked` | An error message that names the actual mismatch |

---

## 1. Parametrized tests: `subTest`

```python
CASES = [(0, False), (1, False), (2, True), (3, True), ...]

def test_table(self):
    for n, expected in self.CASES:
        with self.subTest(n=n):
            self.assertEqual(m.is_prime(n), expected)
```

Look at `TestIsPrime` in `test_exercises.py`. Without `subTest`, the first
failing case in the loop would raise and **stop the whole test method** —
you would fix it, rerun, and find out about the *next* failing case one at
a time. `subTest` catches each iteration's assertion separately: a failing
`n=8` is reported as its own labelled failure, and the loop keeps going
through the rest of the table regardless. One test method, one table, full
visibility into every case at once — table-driven testing, in the
vocabulary this repo uses elsewhere for "the spec as executable tests."

**In `exercises.py`:** `is_prime`.

In [ ]:
def is_prime(n):
    """True if n is prime.

    The function itself is deliberately simple - the point of this
    exercise is test_exercises.py's use of subTest to run it against a
    whole table of (input, expected) cases as one parametrized test,
    rather than one test method per case.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§1", globals())

## 2. Mocking: functions shaped to be testable

```python
def fetch_first_success(client, ids):
    for item_id in ids:
        try:
            result = client.get(item_id)
        except Exception:
            continue
        if result is not None:
            return result
    return None
```

`client` is a parameter, not a module-level `import requests` used
directly inside the function. That one design choice — **dependency
injection** — is what makes this testable without a network:

```python
from unittest.mock import Mock

client = Mock()
client.get.side_effect = [Exception("boom"), None, "found"]
fetch_first_success(client, [1, 2, 3])   # "found"
```

`Mock().get` is a stand-in callable; `side_effect` set to a list makes
each successive call return the next item, or raise it if the item is an
exception instance. `client.get.assert_any_call(10)` afterwards checks
*how* the mock was used, not just what came back — read
`TestFetchFirstSuccess` in `test_exercises.py` for the full pattern,
including asserting the search stops at the first success rather than
continuing.

A function that reaches into a global `requests` module directly has no
such seam — mocking it means monkeypatching the import itself, which is
exactly the awkward, brittle pattern dependency injection avoids.

**In `exercises.py`:** `fetch_first_success`.

In [ ]:
def fetch_first_success(client, ids):
    """Try client.get(id) for each id in order; return the first success.

    Ids whose call raises are skipped, not fatal - one bad id should not
    stop the search. Taking `client` as a parameter (dependency
    injection) rather than importing a concrete HTTP client at module
    level is what makes this testable at all: a test passes a
    unittest.mock.Mock instead of hitting a real network.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§2", globals())

## 3. Injectable clocks: testing time without waiting

```python
class TTLCache:
    def __init__(self, now_fn=time.time):
        self._now = now_fn
```

Same idea as `client` above, applied to `time.time`. A test that wants to
prove an entry expires after 60 seconds has two options: call
`time.sleep(61)` (slow, and still flaky if the process is scheduled out
for a moment near the boundary), or inject a fake clock it controls
exactly:

```python
now = {"t": 0}
cache = TTLCache(now_fn=lambda: now["t"])
cache.set("a", "value", ttl=10)
now["t"] = 11
cache.get("a")   # None - "11 seconds later" took zero wall-clock time
```

This is the general answer to "how do I test something that depends on
the current time" — inject the clock, don't call the real one.

**In `exercises.py`:** `TTLCache.set`, `TTLCache.get`.

In [ ]:
class TTLCache:
    """A cache where entries expire after a time-to-live, in seconds.

    now_fn defaults to time.time but is injectable - exactly like
    `client` above. Time-based logic is notoriously hard to test
    honestly (a test that calls time.sleep(61) to prove an entry expired
    after 60s is slow and still flaky under load); injecting the clock
    lets a test substitute a fake one it controls exactly.
    """

    def __init__(self, now_fn=time.time):
        self._now = now_fn
        self._store = {}

    def set(self, key, value, ttl):
        raise NotImplementedError("your code here")

    def get(self, key):
        raise NotImplementedError("your code here")

In [ ]:
lab.check("§3", globals())

## 4. Logging instead of printing, and `assertLogs`

```python
logger = logging.getLogger(__name__)

def divide_safely(a, b):
    if b == 0:
        logger.warning("division by zero: %r / %r", a, b)
        return None
    return a / b
```

`logger.warning(...)` instead of `print(...)` because logging is
structured and controllable: a caller can set the level, redirect it, or
turn it off, none of which is true of stdout. `unittest.TestCase.assertLogs`
is the matching testing tool — it captures records emitted inside the
`with` block at or above a given level, and **raises `AssertionError` if
nothing was logged**, which is what makes it useful both for "this should
log a warning" and, as its own negative-space test, "this should log
nothing":

```python
with self.assertLogs(level="WARNING") as captured:
    divide_safely(1, 0)
self.assertEqual(captured.records[0].levelno, logging.WARNING)
```

**In `exercises.py`:** `divide_safely`.

In [ ]:
def divide_safely(a, b):
    """Divide a by b; log a warning and return None instead of raising on b == 0.

    logger.warning, not print - a caller (or test) can control or
    capture logging output without the function needing to know how its
    caller wants messages handled. unittest's assertLogs is the
    standard way to assert a specific message was logged at a specific
    level, without capturing stdout.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§4", globals())

## 5. Bisection as a debugging technique

```python
def find_first_failure(n, predicate):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if predicate(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo
```

This is `git bisect`'s algorithm, stripped down to the loop underneath
it. Given a **monotonic** predicate — false for a while, then true from
some point on, never flipping back — bisection finds the exact boundary
in `O(log n)` calls to `predicate` instead of checking every index in
order. `git bisect` applies this to "which commit introduced the bug"
with the predicate being "does this commit's build pass the failing
test"; the same shape applies any time you're narrowing down *where*
something started going wrong, over a range of indices, dates, or
config versions, not just commits.

```python
>>> find_first_failure(1000, lambda i: i >= 743)
743                      # found in ~10 calls, not 743
```

**In `exercises.py`:** `find_first_failure`.

In [ ]:
def find_first_failure(n, predicate):
    """Binary-search range(n) for the first index where predicate(i) is True.

    predicate is assumed MONOTONIC: False for a while, then True from
    some point on (never back to False) - exactly the assumption behind
    `git bisect`. Instead of checking every index in order (O(n), and
    each check might be an expensive test run or a slow query),
    bisection finds the boundary in O(log n) checks. Returns n if
    predicate is False everywhere.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§5", globals())

## 6. Error messages that name the mistake

```python
class MatrixShapeError(ValueError):
    """Raised when two matrices' shapes are incompatible for multiplication."""

def matmul_checked(a, b):
    ...
    if cols_a != rows_b:
        raise MatrixShapeError(
            f"cannot multiply a {rows_a}x{cols_a} matrix by a {rows_b}x{cols_b} "
            f"matrix: {cols_a} (columns of a) != {rows_b} (rows of b)"
        )
```

A bare `IndexError` from the multiply loop itself, or a generic
`"shapes don't match"`, both fail without saying *what* was wrong. This
checks the shapes up front and names **both** of them, plus which
dimensions were compared — the same principle behind every assertion
message in this repo's own test suites (`"membership looks linear -
convert b to a set"` from module 14, for instance). A subclass of
`ValueError` rather than a bare `raise ValueError(...)` also lets a
caller catch specifically `MatrixShapeError` without also swallowing
unrelated value errors.

```python
>>> matmul_checked([[1, 2, 3]], [[1, 2], [3, 4]])
Traceback (most recent call last):
    ...
MatrixShapeError: cannot multiply a 1x3 matrix by a 2x2 matrix: 3 (columns of a) != 2 (rows of b)
```

**In `exercises.py`:** `matmul_checked`.

In [ ]:
def matmul_checked(a, b):
    """Multiply two matrices (lists of lists); raise MatrixShapeError on mismatch.

    A generic "shapes don't match" or an IndexError from the multiply
    loop itself both fail without saying *what* was wrong. Checking the
    shapes up front and naming both of them in the message is the
    difference between a five-second fix and a debugging session - the
    same principle as this repo's own test failure messages.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§6", globals())

---

## The exercises

| Function / class | What it teaches |
|---|---|
| `is_prime` | `subTest` — table-driven tests with per-case visibility |
| `fetch_first_success` | Dependency injection as the thing that makes mocking possible |
| `TTLCache` | Injecting a clock instead of sleeping in tests |
| `divide_safely` | `logging` over `print`; `assertLogs` |
| `find_first_failure` | Bisection as a general debugging technique |
| `matmul_checked` | Error messages that name the actual mismatch |

Every function in this table has a stub in `exercises.py` and test
coverage in `test_exercises.py` — a clean 1:1 mapping.

## Check your understanding

1. **What does `self.subTest(n=n)` buy you over a plain `for` loop with
   no `subTest`?**
   <details><summary>Answer</summary>Without <code>subTest</code>, the
   first failing iteration raises and stops the whole test method —
   later cases in the loop never run that pass. <code>subTest</code>
   catches each iteration's assertion failure separately and reports it
   labelled with the loop variables, so a run shows every failing case
   in the table at once instead of one at a time across repeated
   test-fix-rerun cycles.</details>

2. **Why is `fetch_first_success(client, ids)` easier to test than a
   version that imports a concrete HTTP client at module level and uses
   it directly?**
   <details><summary>Answer</summary>Taking <code>client</code> as a
   parameter means a test can pass a
   <code>unittest.mock.Mock</code> instead — no network call happens,
   and the test controls exactly what each call returns via
   <code>side_effect</code>. A function that reaches for a
   module-level import directly has no such seam; testing it means
   monkeypatching the import itself, which is more brittle.</details>

3. **Why does `TTLCache` take `now_fn` as a parameter instead of calling
   `time.time()` directly inside `set`/`get`?**
   <details><summary>Answer</summary>So a test can substitute a fake
   clock it fully controls, advancing "time" instantly by changing what
   the fake returns — instead of calling <code>time.sleep()</code> for
   real, which is slow and can be flaky under load near a boundary.</details>

4. **What does `unittest.assertLogs` actually assert if nothing inside
   the `with` block calls a logger?**
   <details><summary>Answer</summary>It raises <code>AssertionError</code>
   — <code>assertLogs</code> requires at least one matching log record
   to have been emitted inside the block, so it can also be used
   negatively (wrapped in <code>assertRaises(AssertionError)</code>) to
   prove that a code path does <em>not</em> log anything.</details>

5. **Why does bisection need the predicate to be monotonic, and what
   goes wrong if it isn't?**
   <details><summary>Answer</summary>Bisection decides which half of the
   remaining range to discard based on a single sample at the midpoint,
   trusting that "true" stays true for everything past that point (and
   "false" stays false before it). If the predicate can flip back and
   forth, the half it discards might still contain the boundary it's
   looking for, and the search can return a wrong or arbitrary
   result.</details>

## Interview questions this module answers

- *How would you test a function that calls an external API?*
- *How do you test time-based logic without making the test slow or
  flaky?*
- *What's the advantage of `subTest` over one test method per case, or
  over a loop with no `subTest` at all?*
- *A build has been failing since one of the last 500 commits — how do
  you find which one, fast?*
- *What makes an error message actually useful for debugging, versus
  technically correct?*

## Finished? Check everything, then record it

The first cell checks the whole unit. The second records your result in LinguAgent (needs your lab token from [https://linguagent.com/settings#labs](https://linguagent.com/settings#labs) — in Colab, add it once as a secret named `LINGUAGENT_TOKEN` under 🔑 in the left bar).

In [ ]:
lab.check_all(globals())

In [ ]:
lab.sync()

**Next:** [DL 1 - Tensors](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/ai-dl/22-tensors/lab.ipynb) · [the lesson in LinguAgent](https://linguagent.com/courses/python/testing-and-debugging)